# Update the `year`, the `timeframe` and the `message_end` below to reflect the relevant dates. 

These will impact the Folder, Filenames, the Email Subject & Body, and the BigQuerytable where the results are saved. Examples below:
**Folder:** Shared Documents\Enrichment\Vendor\Vendor Scorecards\2025\Q1
**File:** NESTLE_PCX_ENRICHMENT_SCORECARD_Q1_2025
**Email Subject:"** PCX Supplier Enrichment Scorecar - Nestle Q1-2025
**Email Body:"**  We are pleased to send you your PCX Supplier Enrichment Scorecard for Q1 2025.

In [30]:
# Time period to describe the current round of scorecards
# e.g. 'Q1' for first quarter

timeframe = 'Q2'
year = '2025'

# Last sentence to be used in the message body - reference the next scorecard
#message_end = "The Q3 scorecard will be sent out in September 2025"

# Username of the person running the script (e.g. for Jonathan Gruber-Benaich use 'jongrub')
username = 'jongrub'

# List of addresses that should be CC'd on every email
cc_recipients = 'jacqui.nie@loblaw.ca;'

In [31]:
# Mark the time the script started running

import datetime

starttime = datetime.datetime.now()

print('This script starting running at ' + str(starttime))

This script starting running at 2025-07-03 09:51:45.487249


In [32]:
# command to install relevant packages mentioned below 
# remove comment from any lines where package needs to be re-installed and Run this cell


#pip install PyWin32 
#pip install Office365-REST-Python-Client 

In [33]:
# Identify the folder where the scorecard files were generated

import os
from pathlib import Path

# Set the root folder
root_folder = f'C:\\Users\\{username}\\OneDrive - George Weston Limited-6469347-MTCAD\\Shared Documents\\Enrichment\\Vendor\\Vendor Scorecards'

# Create new folders for this year and timeframe if they don't already exist
curr_dir = f'{root_folder}\\{year}\\{timeframe}'
Path(curr_dir).mkdir(parents=True, exist_ok=True)

# Set the current working directory to the folder for the current year/timeframe
os.chdir(curr_dir)
cwd = os.getcwd()
print('The current working directory is ' + '"' + cwd + '"')

The current working directory is "C:\Users\jongrub\OneDrive - George Weston Limited-6469347-MTCAD\Shared Documents\Enrichment\Vendor\Vendor Scorecards\2025\Q2"


In [34]:
# Connect to BigQuery
from google.cloud import bigquery
import google.auth
project = 'ld-pcx-bia'
conn = bigquery.Client(project=project)

# Run a query to select all contact info for all suppliers

query = f'''
    SELECT * FROM `ld-pcx-bia.Merch_PIM.supplier_scorecard_distribution_v`
    '''

conn.query(
                    query,
                    bigquery.QueryJobConfig(dry_run=True, use_query_cache=False),
                ).total_bytes_billed
query_result = conn.query_and_wait(query)
query_result = conn.query(query)

# Create a dataframe with data for all suppliers
df = query_result.to_dataframe()

# Sort dataframe by the relevant columns
df_sorted = df.sort_values(['rolodex_name'], ignore_index=True)

print(df_sorted.head())

    rolodex_name                                   recipient_emails
0     A Lassonde  akanksha.sharma@lassonde.com; beata.stolarski@...
1        Agropur  Florence.Pigeon@agropur.com; olivier.maltais-s...
2  Aliments Roma                   denisse.sanchez@alimentsroma.com
3   Baby Gourmet                              kevin@babygourmet.com
4        Barilla  gregorio.lopez-bondi@barilla.com; shahzaib.has...


C:\Users\jongrub\Anaconda3\lib\site-packages\google\cloud\bigquery\table.py:2309: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
C:\Users\jongrub\Anaconda3\lib\site-packages\google\cloud\bigquery\table.py:2323: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
C:\Users\jongrub\Anaconda3\lib\site-packages\google\cloud\bigquery\table.py:2337: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
C:\Users\jongrub\Anaconda3\lib\site-packages\google\cloud\bigquery\table.py:1727: UserWarning: BigQuery Storage module not found, fetch dat

In [ ]:
# Use classic Outlook - the new Outlook does not support this action

import win32com.client as win32

# Connect to Outlook
outlook = win32.Dispatch('outlook.application')

# Create a loop to repeat the steps below for each supplier in the table
for index, row in df_sorted.iterrows():
    vendor = row["rolodex_name"]
    recipients = row["recipient_emails"]
    attachment = f"{cwd}\{vendor}_PCX_SUPPLIER_ENRICHMENT_SCORECARD_{timeframe}_{year}.xlsx"
    
    try:
        mail = outlook.CreateItem(0)
        mail.To = recipients
        mail.CC = cc_recipients
        # Subject Line to be used for each email
        mail.Subject = timeframe + " " + year + " " ' PCX Supplier Enrichment Scorecard - ' + vendor
        mail.BodyFormat = 2
        # HTML message body to be used for each email
        mail.HTMLBody = '''
            <html>
            <head></head>
            <body>
            <p>Hello,</p>
            <p>We are pleased to send you your <strong><u>PCX Supplier Enrichment Scorecard</u></strong> for <strong>Q2 2025</strong>.</p>
            <p>As a supplier whose products are sold on PC Express (e.g. Loblaws.ca, nofrills.ca, realcanadiansuperstore.ca), you can now review online product information of all your active products in one convenient place. Our goal is to provide you with visibility to your digital product content on PCX so that you can take actionable steps to improve the information where needed.</p>
            <p>In the scorecard, you will find:</p>
            <ol>
            <li><strong>Scoring &amp; Instructions</strong>: Details on how you are scored and instructions for updating your online product information</li>
            <li><strong>Summary</strong>: Gives you your overall enrichment score and offers a high-level summary of the enrichment status of your products</li>
            <li><strong>Item Details</strong>: Detailed look at all your active articles in our catalog, their individual attributes, and their enrichment status and details</li>
            </ol>
            <p>Please reach out to our team if you have any questions or comments about your scorecard, or refer to our <a href="https://urldefense.com/v3/__https:/docs.google.com/document/d/e/2PACX-1vTyL8GkrErxL9gh-QubXTZ4yKH-EowMikP_m0aVnisVxqBRtcN4u8BHzMGf_eb0-TQ-vm73z5TuOdrP/pub__;!!GUxQW5qUiFhLkzdRaA!bdAqFyFVe4nDUyvgNB3z5dxd9iXO-Hj_iiPjpykjQ5qJ53hnKYPc8VbJSl0GUxWbolQzJHW0whpgZVZqQOynuqgl9zyHZMsPAC4$">Supplier FAQ</a>. <strong>The Q3 2025 scorecard will be sent out by end of September 2025.</strong></p>
            <p><strong><span style="background-color: #ffff00;">Important update &ndash; please read:</span></strong></p>
            <p><strong>PCX is migrating our image management system,</strong> <strong>with completion expected in mid-July</strong>. Post-migration, most of your images should reflect correctly; however, due to differing functionalities of the new system, there may be discrepancies between what you see on site vs. the scorecard.</p>
            <p>For example, after mid-July, you may see some duplicate images or old images for your products mixed in with current images. While we have already cleaned up most of these instances in the backend, please report any discrepancies so we can promptly correct them.</p>
            <p>&nbsp;</p>
            <p>Thank you!</p>
            <p style="font-family:'Helvetica';font-size:10;font-weight:bold">Jonathan Gruber-Benaich</p>
            <p style="font-family:'Helvetica';font-size:9">Manager, Product Information Management</p>
            <p style="font-family:'Helvetica';font-size:9">He/Him</p>
            <p style="font-family:'Helvetica';font-size:9">500 Lake Shore Blvd West, Suite 500</p>
            <p style="font-family:'Helvetica';font-size:9">Toronto, ON M5V 1A5</p>
            </body>
            </html>
        '''
 
        # Attach file (with error handling)
        try:
            mail.Attachments.Add(attachment)
            
        except Exception as e:
            print(f"Error attaching file for {vendor}: {e}")
            continue  # Skip this email and process the next one

        # Save the email (currently saving as draft)
        # Use mail.Send() if you want to send directly
        mail.Save()  

    except Exception as e:
        print(f"Error processing email for {vendor}: {e}")

del outlook

In [36]:
# Mark the time the script stopped running

import datetime

endtime = datetime.datetime.now()

print('This script stopped running at ' + str(endtime))

This script stopped running at 2025-07-03 09:52:04.071810
